In [124]:
from typing import NamedTuple, TypedDict, AnyStr, Callable, Any, TypeVar

import pandas as pd
import requests
from pathlib import Path

from narwhals import DataFrame

from tqdm.notebook import tqdm
import zipfile
import librosa
import IPython
from glob import glob
import numpy as np

from feat_ext import spectral_centroid, root_mean_square, zero_crossing_rate

In [125]:
np.random.seed(0)

In [126]:
# https://stackoverflow.com/a/62113293
def download(url, path: Path):
    resp = requests.get(url, stream=True)
    total = int(resp.headers.get('content-length', 0))
    with open(path, 'wb') as file, tqdm(
            desc=path.name,
            total=total,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
    ) as bar:
        for data in resp.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

In [127]:
download_path = Path("data/dataset.zip")
if not (download_path.exists() and download_path.is_file()):
    download("https://zenodo.org/records/3371780/files/audio_mono-mic.zip?download=1", download_path)

In [128]:
guitar_set_dir = Path("data/GuitarSet")
if not (guitar_set_dir.exists() and guitar_set_dir.is_dir()):
    with zipfile.ZipFile(download_path, "r") as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc="Extracting "):
            zip_ref.extract(member, guitar_set_dir)

In [129]:
# example, sr = librosa.load(guitar_set_dir.joinpath("00_BN1-129-Eb_comp_mic.wav"))
# IPython.display.Audio(data = example, rate=sr)

In [130]:
class Sample(NamedTuple):
    audio: Path
    sr: int
    sample_start: int
    sample_end: int


class SampleWithFeatures(NamedTuple):
    audio: Path
    sr: int
    sample_start: int
    sample_end: int
    zero_crossing_rate: float
    spectral_centroid: float
    rms: float


def sample(s: Sample, zero_crossing_rate: float, spectral_centroid: float, rms: float) -> SampleWithFeatures:
    return SampleWithFeatures(
        audio=s.audio,
        sr=s.sr,
        sample_start=s.sample_start,
        sample_end=s.sample_end,
        zero_crossing_rate=zero_crossing_rate,
        spectral_centroid=spectral_centroid,
        rms=rms
    )


def extract_feature(audio: np.ndarray, y: Sample) -> SampleWithFeatures:
    zcr = zero_crossing_rate(audio)
    sc = spectral_centroid(audio)
    rms_v = root_mean_square(audio)
    return sample(y, zcr, sc, rms_v)


In [131]:
T = TypeVar('T')


def split_into_samples(files: list[AnyStr], sample_length: int,
                       f: Callable[[np.ndarray, Sample], T] = lambda _, y: y) -> list[T]:
    dataset_list: list[T] = []
    for file in (p := tqdm(files)):
        try:
            sample, sr = librosa.load(Path(file))
        except Exception as e:
            print(f"Failed to load {file}: {e}")
            continue
        length: float = len(sample) / sr
        sample_no = int(length // sample_length)
        remainder: float = length - sample_length * sample_no

        last_offset: int = 0


        for i in range(sample_no):
            offset: float = np.random.random() * remainder
            remainder -= max(0, offset)
            start_index = last_offset + int(offset * sr)
            end_index = min(int(last_offset + offset * sr + sample_length * sr), len(sample))

            s = Sample(audio=Path(file), sr=sr, sample_start=start_index, sample_end=end_index)

            p.set_postfix_str(f"{file} {start_index}:{end_index}")

            dataset_list.append(f(sample[start_index:end_index], s))
            last_offset = end_index
    return dataset_list

In [132]:
files = glob(str(guitar_set_dir) + "/**/*", recursive=True)

In [139]:
dataset_5sec: DataFrame = pd.DataFrame(split_into_samples(files, 5, extract_feature))
display(dataset_5sec[dataset_5sec["audio"].apply(str) == "data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])

  0%|          | 0/360 [00:00<?, ?it/s]

,audio,sr,sample_start,sample_end,zero_crossing_rate,spectral_centroid,rms
0,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,13377,123627,0.031819,2846.863218,0.013388
1,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,125000,235250,0.043610,2835.143990,0.011450
2,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,235394,345644,0.041506,3477.637546,0.009217
3,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,346187,456437,0.029715,3733.299332,0.013290
4,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,456595,566845,0.039973,3677.739675,0.012325
5,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,566850,677100,0.038340,3039.376674,0.012121
6,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,677100,787350,0.043973,3404.675203,0.008499
7,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,787350,897600,0.042703,3521.755297,0.011561
8,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,897600,1007850,0.047148,3209.094212,0.009484


In [134]:
dataset_10sec: DataFrame = pd.DataFrame(split_into_samples(files, 10, extract_feature))
display(dataset_10sec[dataset_10sec["audio"].apply(str) == "data/GuitarSet/00_SS3-84-Bb_comp_mic.wav"])

  0%|          | 0/360 [00:00<?, ?it/s]

,audio,sr,sample_start,sample_end,zero_crossing_rate,spectral_centroid,rms
0,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,76146,296646,0.045937,3247.130772,0.011188
1,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,311702,532202,0.034862,3678.519152,0.013048
2,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,552204,772704,0.039923,3040.531100,0.010625
3,data/GuitarSet/00_SS3-84-Bb_comp_mic.wav,22050,775189,995689,0.045170,3439.881990,0.010571


In [135]:
def get_audio(df: DataFrame, i: int) -> tuple[np.ndarray, int]:
    row = df.iloc[i]
    x, sr = librosa.load(row.audio)
    print(row.zero_crossing_rate.shape)
    print(row.rms.shape)
    return x[row.sample_start:row.sample_end], sr

In [136]:
x, sr = get_audio(dataset_10sec, 0)
IPython.display.Audio(x, rate=sr)

()
()


In [137]:
x, sr = get_audio(dataset_5sec, 1)
IPython.display.Audio(x, rate=sr)

()
()


In [138]:
%store get_audio
%store dataset_5sec

Proper storage of interactively declared classes (or instances
of those classes) is not possible! Only instances
of classes in real modules on file system can be %store'd.

Stored 'dataset_5sec' (DataFrame)
